<a href="https://colab.research.google.com/github/rahafabumwise/CV_Assistant/blob/main/CV_Assistant_Phase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/rahafabumwise/CV_Assistant.git /content/CV_Assistant

Cloning into '/content/CV_Assistant'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 57 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 28.90 KiB | 799.00 KiB/s, done.
Resolving deltas: 100% (15/15), done.


In [2]:
%cd /content/CV_Assistant

/content/CV_Assistant


In [3]:
from pathlib import Path
import csv

# Find the documents used for continued pretraining.
documents = sorted(Path("data/cpt/documents").glob("*.txt"))

total_words = 0

for document in documents:
    text = document.read_text(encoding="utf-8")
    word_count = len(text.split())
    total_words += word_count

    print(f"{document.name}: {word_count} words")

# Read the separate evaluation questions.
with open("tests/test_questions.csv", encoding="utf-8", newline="") as file:
    test_questions = list(csv.DictReader(file))

print("\nNumber of documents:", len(documents))
print("Total words:", total_words)
print("Number of test questions:", len(test_questions))

tru_action_verbs.txt: 176 words
tru_education_training_and_certificates.txt: 370 words
tru_employment_or_work_experience.txt: 350 words
tru_hobbies_interest.txt: 114 words
tru_modern_resume_formatting.txt: 1034 words
tru_overall_format.txt: 484 words
tru_transferable_skills.txt: 355 words
tru_volunteering_community_involvement.txt: 183 words

Number of documents: 8
Total words: 3066
Number of test questions: 30


In [4]:
%pip install -q transformers

In [5]:
from transformers import AutoTokenizer
import transformers

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

sentence = "I am writing my first CV."

token_ids = tokenizer.encode(sentence, add_special_tokens=False)

print("Text:", sentence)
print("Token pieces:", [tokenizer.decode([i]) for i in token_ids])
print("Token IDs:", token_ids)
print("Number of tokens:", len(token_ids))
print("Transformers version:", transformers.__version__)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Text: I am writing my first CV.
Token pieces: ['I', ' am', ' writing', ' my', ' first', ' CV', '.']
Token IDs: [40, 716, 3597, 616, 717, 26196, 13]
Number of tokens: 7
Transformers version: 5.22.1


In [6]:
from pathlib import Path

documents = sorted(
    Path("/content/CV_Assistant/data/cpt/documents").glob("*.txt")
)

assert len(documents) == 8, "Expected 8 text files. Check the folder."

document_token_counts = {}
total_tokens = 0

for document in documents:
    text = document.read_text(encoding="utf-8")

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )

    document_token_counts[document.name] = len(token_ids)
    total_tokens += len(token_ids)

    print(f"{document.name}: {len(token_ids):,} tokens")

print(f"\nTotal document tokens: {total_tokens:,}")

tru_action_verbs.txt: 339 tokens
tru_education_training_and_certificates.txt: 445 tokens
tru_employment_or_work_experience.txt: 420 tokens
tru_hobbies_interest.txt: 145 tokens
tru_modern_resume_formatting.txt: 1,379 tokens
tru_overall_format.txt: 600 tokens
tru_transferable_skills.txt: 497 tokens
tru_volunteering_community_involvement.txt: 233 tokens

Total document tokens: 4,058


In [7]:
import json
from pathlib import Path

project_folder = Path("/content/CV_Assistant")
data_folder = project_folder / "data" / "cpt"

# Keep this entire document out of training.
validation_filename = "tru_education_training_and_certificates.txt"

train_files = [
    document
    for document in documents
    if document.name != validation_filename
]

validation_files = [
    document
    for document in documents
    if document.name == validation_filename
]

# Check that the split contains the expected files.
assert len(train_files) == 7
assert len(validation_files) == 1
assert set(train_files).isdisjoint(validation_files)

train_tokens = sum(
    document_token_counts[file.name] for file in train_files
)

validation_tokens = sum(
    document_token_counts[file.name] for file in validation_files
)

# Save the exact filenames so later runs use the same split.
split_information = {
    "train": [
        str(file.relative_to(project_folder))
        for file in train_files
    ],
    "validation": [
        str(file.relative_to(project_folder))
        for file in validation_files
    ],
    "train_tokens_without_separators": train_tokens,
    "validation_tokens_without_separators": validation_tokens,
    "selection_method": "Fixed whole-document holdout",
    "validation_document": validation_filename
}

split_path = data_folder / "split.json"

split_path.write_text(
    json.dumps(split_information, indent=2),
    encoding="utf-8"
)

print("Training documents:", len(train_files))
print("Training tokens:", train_tokens)
print("Validation documents:", len(validation_files))
print("Validation tokens:", validation_tokens)
print("\nSplit saved to:", split_path)

Training documents: 7
Training tokens: 3613
Validation documents: 1
Validation tokens: 445

Split saved to: /content/CV_Assistant/data/cpt/split.json
